# CAMB vs Sachs-Wolfe Comparison

Compares the SW-only $C_\ell$ approximation against full CAMB integration for the same $P_\mathcal{R}(k)$, isolating the ISW contribution at low $\ell$.

**Goal:** Quantify how much the ISW effect changes the low-$\ell$ $\chi^2$ vs Planck 2018.

**Prediction:** ISW adds ~30-50% power at $\ell \lesssim 10$, raising both model and LCDM $D_\ell$ approximately equally, leaving $\Delta\chi^2$ largely unchanged.

In [ ]:
import os, json, glob, sys
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath('..'))

import matplotlib
matplotlib.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "figure.dpi": 300,
    "font.family": "serif",
})

from scripts.camb_wrapper import compute_cl_full_camb, compute_cl_camb_powerlaw, run_comparison, compute_chi2_camb
from scripts.sachs_wolfe import compute_cl_sw, compute_cl_sw_powerlaw
from scripts.planck_data import get_planck_data_asymmetric
from pspectrum_pipeline import load_pspectrum
from scripts.constants import As, k_pivot_phys, T_cmb

TOL = {
    "blue": "#4477AA", "red": "#CC3311", "green": "#228833",
    "yellow": "#EE8866", "teal": "#44BB99", "purple": "#AA3377",
    "grey": "#666666", "dark": "#222222",
}

print("Imports OK")

In [ ]:
ps_dir = "outputs/simulations/pspectra"
files = sorted(glob.glob(os.path.join(ps_dir, "*.json")))
print(f"Found {len(files)} P_S(k) files:")
for f in files[-10:]:
    meta = json.load(open(f))["metadata"]
    print(f"  {os.path.basename(f):55s}  \u03c6\u2080={meta['phi0']:.2f}  y\u2080={meta['y0']:.3f}  N*={meta['N_star']:.1f}")

In [ ]:
PS_FILE = files[-1]
print(f"Using: {os.path.basename(PS_FILE)}")
data = load_pspectrum(PS_FILE)
meta = data['metadata']
print(f"  Model: {meta['model']}, \u03c6\u2080={meta['phi0']:.2f}, y\u2080={meta['y0']:.3f}, N*={meta['N_star']:.1f}")

In [ ]:
# Compute SW, full CAMB, and LCDM baseline
print("SW-only...")
ells_sw, C_sw = compute_cl_sw(data, ell_max=30)

print("Full CAMB (low-l)...")
ells_camb, C_camb, C_camb_TE, C_camb_EE = compute_cl_full_camb(data, ell_max=30)

print("Full CAMB (high-l)...")
ells_hi, C_hi, _, _ = compute_cl_full_camb(data, ell_max=2500)

print("LCDM baseline (CAMB)...")
ells_lcdm, C_lcdm, _, _ = compute_cl_camb_powerlaw(ell_max=2500)

def to_Dell(ells, C):
    return C * ells * (ells + 1) / (2 * np.pi) * (T_cmb * 1e6) ** 2

D_sw = to_Dell(ells_sw, C_sw)
D_camb = to_Dell(ells_camb, C_camb)
D_hi = to_Dell(ells_hi, C_hi)
D_lcdm = to_Dell(ells_lcdm, C_lcdm)

p_ells, D_planck, D_err_lo, D_err_hi = get_planck_data_asymmetric()
print("Done.")

In [ ]:
# Chi2 comparison: SW-only vs full CAMB
chi2_camb_m, chi2_camb_l, dchi2_camb = compute_chi2_camb(data)

# SW chi2 (reproduce from sachs_wolfe)
from scripts.sachs_wolfe import _interpolate_cl_to_data
from scripts.planck_data import get_planck_data

planck_ells, D_planck_sym, D_err_sym = get_planck_data()
D_sw_interp = np.interp(planck_ells, ells_sw, D_sw)
D_lcdm_sw_interp = np.interp(planck_ells, ells_sw, to_Dell(*compute_cl_sw_powerlaw(ell_max=30)[:2]))

def chi2_fn(D_model, D_data, err):
    return np.sum((D_model - D_data)**2 / err**2)

chi2_sw_m = chi2_fn(D_sw_interp, D_planck_sym, D_err_sym)
chi2_sw_l = chi2_fn(D_lcdm_sw_interp, D_planck_sym, D_err_sym)
dchi2_sw = chi2_sw_m - chi2_sw_l

print(f"SW-only:   \u03c7\u00b2 = {chi2_sw_m:.2f}  LCDM \u03c7\u00b2 = {chi2_sw_l:.2f}  \u0394\u03c7\u00b2 = {dchi2_sw:+.2f}")
print(f"Full CAMB: \u03c7\u00b2 = {chi2_camb_m:.2f}  LCDM \u03c7\u00b2 = {chi2_camb_l:.2f}  \u0394\u03c7\u00b2 = {dchi2_camb:+.2f}")
print(f"ISW shifts \u0394\u03c7\u00b2 by: {dchi2_camb - dchi2_sw:+.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(3.5, 2.8))

ax.errorbar(p_ells, D_planck, yerr=[D_err_hi, D_err_lo],
            fmt="o", color=TOL["dark"], capsize=3, capthick=1,
            markersize=4, elinewidth=1, label="Planck 2018", zorder=5)

ax.semilogy(ells_sw, D_sw, "s-", color=TOL["red"], lw=1.5, ms=4,
            label="SW-only", zorder=3)
ax.semilogy(ells_camb, D_camb, "o-", color=TOL["blue"], lw=1.5, ms=4,
            label="Full CAMB", zorder=4)
ax.semilogy(ells_lcdm[:29], D_lcdm[:29], "--", color=TOL["grey"], lw=1.2,
            label="LCDM (CAMB)", zorder=2)

ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$D_\ell^{TT}\ [\mu {\rm K}^2]$")
ax.legend()
ax.grid(True, alpha=0.25, which="both")
ax.set_xlim(1.5, 31)

fig.tight_layout()
for ext in ["png", "pdf"]:
    fig.savefig(f"outputs/plots/powerloss/camb_vs_sw_low_ell.{ext}", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
D_isw = D_camb - D_sw
D_isw_pct = D_isw / D_sw * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(3.5, 4.5), sharex=True,
                                gridspec_kw={"height_ratios": [2, 1]})

ax1.bar(ells_sw, D_isw, color=TOL["teal"], alpha=0.7, width=0.8)
ax1.axhline(0, color=TOL["dark"], lw=0.5)
ax1.set_ylabel(r"$\Delta D_\ell^{\rm ISW}\ [\mu {\rm K}^2]$")
ax1.grid(True, alpha=0.25, which="both")

ax2.plot(ells_sw, D_isw_pct, "o-", color=TOL["teal"], lw=1.5, ms=4)
ax2.axhline(0, color=TOL["dark"], lw=0.5)
ax2.set_xlabel(r"$\ell$")
ax2.set_ylabel(r"ISW / SW [%]")
ax2.grid(True, alpha=0.25, which="both")
ax2.set_xlim(1.5, 31)

fig.tight_layout(h_pad=0.5)
for ext in ["png", "pdf"]:
    fig.savefig(f"outputs/plots/powerloss/isw_contribution.{ext}", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))

ax.semilogy(ells_hi, D_hi, "-", color=TOL["red"], lw=1.2,
            label="Higgs USR (full CAMB)")
ax.semilogy(ells_lcdm, D_lcdm, "--", color=TOL["dark"], lw=1.2, alpha=0.6,
            label="LCDM (power-law)")

ax.set_xlabel(r"$\ell$", fontsize=14)
ax.set_ylabel(r"$D_\ell^{TT}\ [\mu {\rm K}^2]$", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25, which="both")
ax.set_xlim(1.5, 2500)

fig.tight_layout()
for ext in ["png", "pdf"]:
    fig.savefig(f"outputs/plots/powerloss/camb_full_sky.{ext}", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from scipy.interpolate import interp1d

D_sw_at_p = interp1d(ells_sw, D_sw, kind='cubic')(p_ells)
D_camb_at_p = interp1d(ells_camb, D_camb, kind='cubic')(p_ells)
D_lcdm_at_p = interp1d(ells_lcdm, D_lcdm, kind='cubic')(p_ells)

res_sw = (D_sw_at_p - D_planck) / D_planck * 100
res_camb = (D_camb_at_p - D_planck) / D_planck * 100
res_lcdm = (D_lcdm_at_p - D_planck) / D_planck * 100

fig, ax = plt.subplots(figsize=(3.5, 2.5))

ax.axhline(0, color=TOL["dark"], lw=0.5)
ax.plot(p_ells, res_sw, "s-", color=TOL["red"], lw=1.5, ms=5, label="SW-only")
ax.plot(p_ells, res_camb, "o-", color=TOL["blue"], lw=1.5, ms=5, label="Full CAMB")
ax.plot(p_ells, res_lcdm, "d--", color=TOL["grey"], lw=1.2, ms=4, label="LCDM")

ax.set_xlabel(r"$\ell$")
ax.set_ylabel(r"$(D_\ell - D_\ell^{\rm Planck}) / D_\ell^{\rm Planck}\ [\%]$")
ax.legend()
ax.grid(True, alpha=0.25, which="both")

fig.tight_layout()
for ext in ["png", "pdf"]:
    fig.savefig(f"outputs/plots/powerloss/camb_vs_sw_residuals.{ext}", dpi=300, bbox_inches="tight")
plt.show()

## Summary

- **ISW contribution:** Quantified at each $\ell$ as $\Delta D_\ell = D_\ell^{\rm full} - D_\ell^{\rm SW}$.
- **Effect on $\Delta\chi^2$:** SW-only vs full CAMB comparison shows whether ISW adds a constant shift or changes the relative fit.
- **Full-sky consistency:** Both CAMB and LCDM match the known acoustic peak structure.